# DistilBERT

In [2]:
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (DistilBertTokenizer, DistilBertModel,
                          get_linear_schedule_with_warmup)
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# фиксируем сиды — воспроизводимость (инициализация головы, dropout, шафл)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# GPU если есть: cuda -> mps (Apple) -> cpu; всё (модель и батчи) кладём сюда
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

device: mps


In [3]:
# гиперпараметры в одном месте
MODEL_NAME   = "distilbert-base-uncased"
MAX_LEN      = 128       # обрезаем/дополняем каждый коммент до стольких токенов
BATCH_SIZE   = 16        # BERT тяжёлый — батч маленький
EPOCHS       = 3
LR           = 2e-5      # для дообучения трансформера нужен очень маленький lr
WARMUP_RATIO = 0.1       # первые 10% шагов — разогрев lr
VAL_SIZE     = 0.1

Path("../checkpoints").mkdir(parents=True, exist_ok=True)
Path("../submissions").mkdir(parents=True, exist_ok=True)

# Данные

In [4]:
# для BERT текст НЕ чистим вручную — его токенайзер сам нормализует и приводит к lower (uncased)
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
train["comment_text"] = train["comment_text"].fillna("")
test["comment_text"] = test["comment_text"].fillna("")
print("train:", train.shape, "| test:", test.shape)
train.head(3)

train: (159571, 8) | test: (153164, 2)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0


In [5]:
# делим train/val 90/10
tr_idx, va_idx = train_test_split(np.arange(len(train)), test_size=VAL_SIZE,
                                  random_state=SEED, shuffle=True)
tr = train.iloc[tr_idx].reset_index(drop=True)
va = train.iloc[va_idx].reset_index(drop=True)
print("train:", len(tr), "| val:", len(va))

train: 143613 | val: 15958


# Токенайзер

In [6]:
# токенайзер DistilBERT: режет текст на сабворды и мапит в id из словаря модели
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

# смотрим как это выглядит: [CLS] ... текст ... [SEP], + attention_mask (1 — реальный токен, 0 — pad)
enc = tokenizer("you are an idiot!", max_length=MAX_LEN,
                padding="max_length", truncation=True)
print("ids :", enc["input_ids"][:12])
print("toks:", tokenizer.convert_ids_to_tokens(enc["input_ids"][:12]))
print("mask:", enc["attention_mask"][:12])

ids : [101, 2017, 2024, 2019, 10041, 999, 102, 0, 0, 0, 0, 0]
toks: ['[CLS]', 'you', 'are', 'an', 'idiot', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
mask: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]


# Dataset / DataLoader

In [7]:
# Dataset токенизирует один коммент на лету и отдаёт (input_ids, attention_mask[, label])
class ToxicDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = None if labels is None else torch.as_tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], max_length=MAX_LEN, padding="max_length",
                        truncation=True, return_tensors="pt")
        input_ids = enc["input_ids"].squeeze(0)          # (MAX_LEN,)
        attention_mask = enc["attention_mask"].squeeze(0)
        if self.labels is None:
            return input_ids, attention_mask              # тест: без меток
        return input_ids, attention_mask, self.labels[i]

# метки (N, 6) float32 — BCELoss сравнивает вероятность с целью как float
y_tr = tr[LABELS].values.astype(np.float32)
y_va = va[LABELS].values.astype(np.float32)

# DataLoader выдаёт батчами; shuffle на train — чтобы модель не учила порядок
train_loader = DataLoader(ToxicDataset(tr["comment_text"], y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ToxicDataset(va["comment_text"], y_va), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(ToxicDataset(test["comment_text"]), batch_size=BATCH_SIZE, shuffle=False)

ids, mask, lab = next(iter(train_loader))
print("батч:", ids.shape, mask.shape, lab.shape)

батч: torch.Size([16, 128]) torch.Size([16, 128]) torch.Size([16, 6])


# Модель

In [ ]:
class ToxicBert(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_labels=len(LABELS), dropout=0.3):
        super().__init__()
        # предобученный трансформер; дообучаем его веса целиком (fine-tuning)
        self.bert = DistilBertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)              # регуляризация, в eval() выключается
        self.fc = nn.Linear(768, num_labels)            # 768 — размер скрытого состояния DistilBERT

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # [CLS] — служебный токен в начале каждой последовательности; его финальное скрытое
        # состояние агрегирует смысл всего коммента, поэтому берём его как вектор предложения
        cls = out.last_hidden_state[:, 0, :]            # (batch, 768)
        x = self.dropout(cls)
        logits = self.fc(x)                             # (batch, 6) сырые скоры
        # sigmoid, а не softmax: метки независимы, у коммента может быть несколько сразу
        return torch.sigmoid(logits)

model = ToxicBert().to(device)                          # кладём параметры на device
print("параметров:", sum(p.numel() for p in model.parameters()))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


параметров: 66367494


In [9]:
criterion = nn.BCELoss()

# AdamW, а НЕ Adam: AdamW корректно отделяет weight decay (L2) от адаптивного шага.
# В обычном Adam регуляризация искажается адаптивным lr — для трансформеров это стандарт.
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# Линейный шедулер с разогревом: lr линейно растёт 0 -> LR за первые 10% шагов, затем
# линейно падает к 0. Разогрев нужен, т.к. в начале градиенты шумные, а большой lr сразу
# затрёт предобученные веса BERT. Плавный старт стабилизирует дообучение.
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print("шагов всего:", total_steps, "| разогрев:", warmup_steps)

шагов всего: 26928 | разогрев: 2692


# Обучение

In [10]:
# eval: model.eval() выключает dropout (детерминированный проход), no_grad() не строит граф
# (не нужен backward) -> экономия памяти и времени на инференсе
@torch.no_grad()
def evaluate(loader, has_labels=True):
    model.eval()
    probs_all, targets_all, losses = [], [], []
    for batch in loader:
        if has_labels:
            ids, mask, y = batch
            ids, mask, y = ids.to(device), mask.to(device), y.to(device)
            probs = model(ids, mask)
            losses.append(criterion(probs, y).item())
            targets_all.append(y.cpu().numpy())
        else:
            ids, mask = batch
            probs = model(ids.to(device), mask.to(device))
        probs_all.append(probs.cpu().numpy())   # обратно на cpu для numpy/sklearn
    probs = np.concatenate(probs_all)
    targets = np.concatenate(targets_all) if has_labels else None
    loss = float(np.mean(losses)) if has_labels else None
    return loss, probs, targets

# ROC-AUC, усреднённый по 6 столбцам (метрика Kaggle)
def mean_auc(targets, probs):
    aucs = []
    for j in range(len(LABELS)):
        if len(np.unique(targets[:, j])) < 2:   # столбец из одних нулей -> AUC не определён
            aucs.append(float("nan"))
        else:
            aucs.append(roc_auc_score(targets[:, j], probs[:, j]))
    return float(np.nanmean(aucs)), aucs

In [11]:
best_auc = -1.0
ckpt_path = "../checkpoints/bert_best.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()                       # режим обучения: dropout включён
    running, t0 = 0.0, time.time()
    for ids, mask, y in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}", leave=False):
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        optimizer.zero_grad()           # обнуляем градиенты — PyTorch их накапливает
        probs = model(ids, mask)        # прямой проход
        loss = criterion(probs, y)      # насколько ошиблись
        loss.backward()                 # обратное распространение: считает .grad всех весов
        optimizer.step()                # шаг оптимизатора: обновляет веса
        scheduler.step()                # двигаем lr по расписанию (разогрев/спад)
        running += loss.item() * ids.size(0)
    train_loss = running / len(tr)

    # валидация после каждой эпохи
    val_loss, val_probs, val_targets = evaluate(val_loader)
    val_auc, _ = mean_auc(val_targets, val_probs)
    print(f"epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
          f"val_auc={val_auc:.4f} ({time.time()-t0:.0f}s)")

    # сохраняем лучший чекпойнт — state_dict (только веса), рекомендованный способ
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({"model_state": model.state_dict(), "val_auc": best_auc}, ckpt_path)
        print(f"  сохранил лучший -> {ckpt_path} (auc {best_auc:.4f})")

print("лучший val AUC:", round(best_auc, 4))

epoch 1/3:   0%|          | 0/8976 [00:00<?, ?it/s]

epoch 1: train_loss=0.0638 val_loss=0.0378 val_auc=0.9910 (6151s)
  сохранил лучший -> ../checkpoints/bert_best.pt (auc 0.9910)


epoch 2/3:   0%|          | 0/8976 [00:00<?, ?it/s]

epoch 2: train_loss=0.0337 val_loss=0.0374 val_auc=0.9920 (7375s)
  сохранил лучший -> ../checkpoints/bert_best.pt (auc 0.9920)


epoch 3/3:   0%|          | 0/8976 [00:00<?, ?it/s]

epoch 3: train_loss=0.0257 val_loss=0.0388 val_auc=0.9915 (5617s)
лучший val AUC: 0.992


# Оценка

In [12]:
# грузим лучший чекпойнт и смотрим AUC по каждой метке
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])

_, val_probs, val_targets = evaluate(val_loader)
val_auc, per_label = mean_auc(val_targets, val_probs)
for name, a in zip(LABELS, per_label):
    print(f"{name:14s}: {a:.4f}")
print(f"{'MEAN':14s}: {val_auc:.4f}")

toxic         : 0.9880
severe_toxic  : 0.9917
obscene       : 0.9937
threat        : 0.9975
insult        : 0.9898
identity_hate : 0.9915
MEAN          : 0.9920


# Сабмит

In [ ]:
_, test_probs, _ = evaluate(test_loader, has_labels=False)
submission = pd.DataFrame(test_probs, columns=LABELS)
submission.insert(0, "id", test["id"].values)
submission.to_csv("../submissions/submission_bert.csv", index=False)
print("submission:", submission.shape)
submission.head()

submission: (153164, 7)


,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,0.995340,0.504153,0.980107,0.092106,0.945126,0.609338
1,0000247867823ef7,0.000878,0.000214,0.000247,0.000194,0.000302,0.000215
2,00013b17ad220c46,0.001107,0.000198,0.000268,0.000146,0.000304,0.000200
3,00017563c3f7919a,0.000455,0.000278,0.000222,0.000240,0.000379,0.000282
4,00017695ad8997eb,0.001849,0.000166,0.000211,0.000179,0.000333,0.000213
